[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/connections/connections.ipynb)

# Connections on Curved Surfaces

This example computes connections on curved surfaces: rules for carrying a tangent vector from each point of a surface to its neighbours. The surface's own connection, parallel transport, keeps the vector tangent and turns it no more than the surface itself bends; from it come the holonomy of loops, the connection Laplacian, vector heat diffusion and the smoothest tangent fields. An extra turn on every edge gives the trivial connections of Crane, Desbrun and Schröder, which design tangent vector fields with their singularities exactly where they are wanted, and the advance of a phase along such a field gives the stripe patterns of Knöppel, Crane, Pinkall and Schröder.

The computation is written in the geometric algebra of three-dimensional space. Between two neighbouring points with unit normals `n_tail` and `n_head`, the shortest turn that carries the first normal onto the second is the square root of their geometric product, `(n_head * n_tail).square_root()`, which turns in the plane of the two normals and leaves the hinge line between their tangent planes still. Chained around a closed curve, those rotors turn a tangent vector by the solid angle the normals sweep out; placed into a *sparse extensor* across the edges of a triangle mesh, they form the covariant derivative and the connection Laplacian directly on ambient three-dimensional vectors, with no local two-dimensional coordinate charts or reference edges.

In the notation of differential geometry, carrying a vector along a curve reads as solving $\nabla_{\dot\gamma} X = 0$ with the Levi-Civita connection $\nabla$, the turn around a loop bounding a region $\Omega$ as $\int_\Omega K\,dA$, and the connection Laplacian as $\nabla^*\nabla$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import concatenate, stack
from numga.algebras import VGA3D as ga
from numga.sparse import SparseExtensor, spdiag
from examples.util.animation import save_figure, show_animation
from examples.surfaces.connections import render
from examples.surfaces.connections.core import latitude_rings
from examples.surfaces.connections.scenarios import face_indices, nearest_faces
from examples import instantiate

Scalar = ga.gatype.scalar()                         # Scalar+
# Positions and tangent vectors are points and directions, antivectors.
Point = ga.gatype.antivector()                      # Antivector-
# The unit normals: the patches of the tangent planes, joins of two edges.
Vector = ga.gatype.vector()                         # Vector-
Rotor = ga.gatype.rotor()                           # Rotor+

mesh_module = instantiate("examples.util.mesh", ga)
Mesh, as_scalar, context = mesh_module.Mesh, mesh_module.as_scalar, mesh_module.ctx
mv = context.multivector

## 1. Holonomy around latitude circles

On the unit sphere each point's outward normal is the point itself. A tangent vector carried around a latitude circle, each step turned by the shortest rotor between neighbouring normals, `(n_head * n_tail).square_root()`, stays tangent all the way, and comes back turned by the solid angle of the cap the circle bounds. On the green equator the cap is a hemisphere, a whole turn, and the vector returns as it started; on the blue circle sixty degrees from the pole the cap's solid angle is π, and the vector returns reversed; on the red circle near the pole the cap is small, and the vector turns by about fifty degrees. The faint arrows trail where each vector has been.

In [ ]:
# The sphere's subdivisions, and its three latitude circles, 30, 60 and 90 degrees from the north pole,
# with the steps around each and the steps between frames.
n_subdivisions = 3
colatitudes = np.array([np.pi / 6, np.pi / 3, np.pi / 2])
n_ring_steps = 210
ring_stride = 2
# The duration of a frame, in every animation.
duration_ms = 40

In [ ]:
def transported(normals: Vector, start: Point) -> Iterator[Point]:
    """Parallel transport carries a tangent vector along a sequence of unit normals `[steps + 1, ...]`, and yields
    it at each of them."""
    yield start
    for tail, head in zip(normals[:-1], normals[1:]):
        start = (head * tail).square_root() >> start                        # [...] Point-
        yield start


sphere_mesh, sphere_vertices = Mesh.icosphere(n_subdivisions)
rings, starts = latitude_rings(colatitudes, n_ring_steps)                  # [steps + 1, rings] Vector-, [rings] Point-
frames = islice(zip(rings, transported(rings, starts)), None, None, ring_stride)
show_animation(render.animate_holonomy(sphere_mesh, sphere_vertices, rings, frames), "connections_holonomy", duration_ms)

## 2. The covariant derivative and connection Laplacian

On a triangle mesh a connection is a rotor on every edge, saying how a tangent vector at the tail is carried into the tangent plane at the head; the surface's own is the shortest turn between the two normals, as on the sphere. The covariant derivative compares, across every edge, the head's vector with the tail's vector carried over: a sparse map from vector fields on the vertices to vectors on the edges, each edge's row holding one at its head and minus the rotor's action at its tail. Weighted by the cotangent ratios of the edges and composed with its adjoint, it gives the connection Laplacian, `covariant.adjoint()(spdiag(weights) * covariant)`, a sparse map between vertex fields of tangent vectors. The tangent vectors are directions, antivectors, intrinsic like all geometry; the normals are the patches of the tangent planes, and a normal's dual is the plane a tangent vector turns in. Every connection further on, with extra turns on its edges, is built by the same function from its own rotors. Because each rotor carries the tail's normal onto the head's, the Laplacian never mixes tangent and normal components.

In the complex line-bundle notation of discrete differential geometry, each vertex picks an arbitrary reference direction in its tangent plane, each edge rotor reads as a unit complex phase $e^{i\theta_{ij}}$ relating the two reference directions, and the connection Laplacian reads as a Hermitian matrix over complex vertex scalars.

In [ ]:
# The torus: its radii, and the vertices around its hole and around its tube.
major_radius = 2.0
minor_radius = 0.75
n_major = 28
n_minor = 14

In [ ]:
def vertex_normals(mesh: Mesh, vertices: Point) -> Vector:
    """The unit normal at each vertex is the sum of its faces' normals, each the join of two of the face's edges,
    twice the face's area large."""
    edges = mesh.boundary[2, 1] * (mesh.d[0] * vertices)                       # [3] Point-[faces]
    return (~mesh.incidence[2, 0].sum(axis=0) * (edges[0] & edges[1])).normalized()  # Vector-[vertices]


def vertex_areas(mesh: Mesh, vertices: Point) -> Scalar:
    """Each vertex's area is a third of the area of each face at it, a face's area half the size of its plane."""
    edges = mesh.boundary[2, 1] * (mesh.d[0] * vertices)                       # [3] Point-[faces]
    corner_mean = mesh.incidence[2, 0].mean(axis=0)                            # Scalar+[faces <- vertices]
    return ~corner_mean * (edges[1].commutator(edges[0]).norm() / 2)           # Scalar+[vertices]


def cotangent_weights(mesh: Mesh, vertices: Point, face_weights: Scalar) -> Scalar:
    """Each edge's weight is half the cotangents of the angles facing it, each face's share weighted by
    `Scalar+[faces]`: the cotangent at a corner is one edge's slice read along the other, over the size of the
    plane they span; head to tail, the corner's own edges point the other way."""
    edges = mesh.boundary[2, 1] * (mesh.d[0] * vertices)                       # [3] Point-[faces]
    after, before = edges[[1, 2, 0]], edges[[2, 0, 1]]                         # [3] Point-[faces], Point-[faces]
    cotangents = -(after.dual_inverse() & before) / after.commutator(before).norm()   # [3] Scalar+[faces]
    return (~mesh.incidence[2, 1] * (face_weights * cotangents / 2)).sum(axis=0)   # Scalar+[edges]


def connection(mesh: Mesh, transports: Rotor, weights: Scalar) -> tuple[SparseExtensor, SparseExtensor]:
    """The covariant derivative of the rotors `[2] Rotor+[edges]` from each end of each edge to its head comes with
    its connection Laplacian, each edge weighted by `Scalar+[edges]`."""
    # Each end's vector carried to the head, signed by the edge's boundary: the head's less the tail's.
    covariant = (spdiag(transports >> Point) * mesh.boundary[1, 0]).sum(axis=0)   # (Point- <- Point-)[edges <- vertices]
    laplacian = covariant.adjoint()(spdiag(weights) * covariant)           # (Point- <- Point-)[vertices <- vertices]
    return covariant, laplacian


mesh, vertices = Mesh.torus(major_radius, minor_radius, n_major, n_minor)
torus_normals = vertex_normals(mesh, vertices)                             # Vector-[vertices]
torus_areas = vertex_areas(mesh, vertices)                                 # Scalar+[vertices]
torus_weights = cotangent_weights(mesh, vertices, as_scalar(np.ones(len(mesh.faces))).field())   # Scalar+[edges]
normals = torus_normals.batch()                                            # [vertices] Vector-

ends = (mesh.incidence[1, 0] * torus_normals)   # [2] Vector-[edges]
# From each end of an edge to its head: the dihedral rotor at the tail, and 1 at the head.
turns = (ends[1] * ends).square_root()                                     # [2] Rotor+[edges]
covariant, laplacian = connection(mesh, turns, torus_weights)              # (Point- <- Point-)[edges <- vertices], [vertices <- vertices]
area_weights = spdiag(torus_areas)                                         # Scalar+[vertices <- vertices]

print(covariant.gatype, covariant.shape)
print(laplacian.gatype, laplacian.shape)

## 3. Parallel transport by vector heat diffusion

A tangent bump on the outer equator of the torus diffuses across the surface by backward-Euler heat steps of the connection Laplacian, carried along every path by the connection as it spreads. Normalized at every vertex, the diffused vector is a unit tangent field carried smoothly around the torus; the animation shows that orientation with symmetric streaks of equal length, independently of the heat amplitude.

In [ ]:
# The bump's width, the heat steps' duration, and their number.
source_width = 0.55
step_time = 0.1
n_heat_steps = 36

In [ ]:
def diffusing(source: Point, n_steps: int) -> Iterator[Point]:
    """Backward-Euler heat steps of the connection Laplacian yield the field after each."""
    step_operator = area_weights * Point + step_time * laplacian           # (Point- <- Point-)[vertices <- vertices]
    for _ in range(n_steps):
        source = step_operator.solve(area_weights * source)                # Point-[vertices]
        yield source


centre = (mv.x * (major_radius + minor_radius)).dual()                # [] Point-
weight = ((vertices.batch() - centre).norm_squared() * (-0.5 / source_width**2)).exp()   # [vertices] Scalar+
direction = (mv.y + mv.z).dual().normalized()                              # [] Point-
# The direction less its part along each normal: the normal's direction times its pairing with the patch.
source = (weight * (direction - normals.dual() * (normals & direction))).field()   # Point-[vertices]
diffused = stack(tuple(diffusing(source, n_heat_steps)))                   # [steps] Point-[vertices]
transported_field = diffused[-1] / diffused[-1].norm()                     # Point-[vertices]

show_animation(render.animate_diffusion(mesh, vertices, diffused), "connections_diffusion", duration_ms)

## 4. Harmonic tangent vector fields

Because a constant normal field also has zero covariant derivative, a stiff penalty along the normals, `spdiag((unit_normals.dual() * (unit_normals & Point)).field()) * normal_stiffness`, lifts the normal bundle out of the low spectrum without touching tangent vectors. The generalized eigenproblem against the vertex areas, `(laplacian + normal_penalty).eigh(area_weights * Point, count)`, then finds the smoothest global tangent fields of any connection Laplacian. Turning every tangent vector a quarter turn in its own tangent plane, its commutator with the plane, `vector.commutator(normals.dual())`, commutes with every edge rotor, so each eigenvalue comes in an equal pair. The figure shows the unit heat-transported field, the fundamental harmonic field and a higher mode.

In [ ]:
# The weight keeping fields out of the normal bundle, and the modes found.
normal_stiffness = 100.0
n_modes = 4

In [ ]:
def low_modes(laplacian: SparseExtensor, normals: Vector, areas: Scalar, normal_stiffness: float, count: int
              ) -> tuple[Scalar, Point]:
    """The `count` smoothest tangent fields of a connection Laplacian against the vertex areas, `Scalar+[vertices]`,
    come with their energies, the normal bundle along the unit normals, `Vector-[vertices]`, lifted out of the low
    spectrum."""
    unit_normals = normals.batch()                                          # [vertices] Vector-
    # A direction's part along the normal: the normal's direction, its dual, times the direction's pairing with
    # the normal's patch.
    normal_penalty = spdiag((unit_normals.dual() * (unit_normals & Point)).field()) * normal_stiffness   # (Point- <- Point-)[vertices <- vertices]
    return (laplacian + normal_penalty).eigh(spdiag(areas) * Point, count)    # [count] Scalar+, [count] Point-[vertices]


mode_energies, modes = low_modes(laplacian, torus_normals, torus_areas, normal_stiffness, n_modes)

save_figure(render.draw_torus_fields(mesh, vertices, [transported_field, modes[0], modes[2]]), "connections_modes");

## 5. The curvature inside a face as a turn

A tangent vector field on a closed surface must vanish or spin somewhere, unless the surface is a torus: a hair on every point of a ball cannot all lie flat, the hairy ball theorem. Each place where a field vanishes has an index, how many times the field turns as one walks once around it, and the indices always add up to a number fixed by the surface's shape, its Euler characteristic: two for a sphere, zero for a torus. The trivial connections of Crane, Desbrun and Schröder design such fields: any choice of places and indices that adds up to that number has a field, and the smoothest one is found by one linear solve.

Carried by parallel transport around the three edges of a triangle and back to the start, a tangent vector comes back turned in its tangent plane, because the surface is curved inside the triangle: the product of the three rotors is a turn about the normal, by the face's holonomy. On a sphere the holonomy of a face is the solid angle its three normals span, and the holonomies of all faces add up to `2 pi` times the Euler characteristic, two whole turns, the discrete theorem of Gauss and Bonnet. A whole turn of a rotor is minus one, so the angle of a loop is read from the square of its rotor, `(loop * loop).log() | normal.dual()`, an angle between minus and plus half a turn.

In [ ]:
def loop_turn(normals: Vector) -> Scalar:
    """The holonomy of transport around closed loops of unit normals `[steps, ...]` is read from the square of the
    running product of the steps."""
    following = concatenate([normals[1:], normals[:1]])                         # [steps, ...] Vector-
    steps = (following * normals).square_root()                                  # [steps, ...] Rotor+
    # The running product of the steps, each later step multiplying from the left.
    around = steps.cumprod(axis=0)[-1]                                           # [...] Rotor+
    return (around * around).log() | normals[0].dual()                           # [...] Scalar+


sphere_normals = vertex_normals(sphere_mesh, sphere_vertices)                         # Vector-[vertices]
sphere_areas = vertex_areas(sphere_mesh, sphere_vertices)                             # Scalar+[vertices]
sphere_weights = cotangent_weights(sphere_mesh, sphere_vertices, as_scalar(np.ones(len(sphere_mesh.faces))).field())   # Scalar+[edges]
corner_normals = (sphere_mesh.incidence[2, 0] * sphere_normals).batch()   # [corners, faces] Vector-
curvature = loop_turn(corner_normals)                                                 # [faces] Scalar+

print("the faces' holonomies, in whole turns:", (curvature.sum(axis=0) / (2 * np.pi)).to_array())

## 6. An extra turn on every edge

A connection is trivial when carrying a vector around any loop returns it unchanged: then carrying one vector out from anywhere gives a field that does not turn at all along the surface. The surface's own transport is not trivial, because the surface is curved, so each edge gets an extra turn by some angle in the head's tangent plane, `(n_head.dual() * (-angle / 2)).exp()`. A turn in a tangent plane commutes with transport into that plane, so the holonomy of a loop grows by exactly the sum of the extra turns along it, each counted forwards or backwards by the direction in which the loop runs along the edge: the circulation of the edge angles around the loop.

Each face is asked to turn by `2 pi` times its chosen index, its own curvature included, so the circulation of the extra turns around every face must be `2 pi` times its index less its curvature: a sparse linear system of one row per face, coupling each face to its three edges. The smallest edge angles that meet it, by least squares, also turn the field the least. A face with index one turns a vector once as one walks around it: the field flows out from it, into it, or around it.

In [ ]:
def adjusted_transport(mesh: Mesh, normals: Vector, angles: Scalar) -> Rotor:
    """The rotors from each end of each edge to its head, `[2] Rotor+[edges]`, are the surface's own transport from
    the tail followed by the edge's extra turn in the head's tangent plane, and none at the head."""
    tails, heads = (mesh.incidence[1, 0] * normals)   # Vector-[edges], Vector-[edges]
    extra = (heads.dual() * (-angles / 2)).exp()                                 # Rotor+[edges]
    forward = extra * (heads * tails).square_root()                              # Rotor+[edges]
    return stack([forward, mv.rotor().broadcast_to(forward.batch().shape).field()])


def trivial_connection(mesh: Mesh, normals: Vector, indices: np.ndarray, generators: list[np.ndarray]
                       ) -> tuple[Scalar, SparseExtensor, Scalar]:
    """The smallest extra turns on the edges make every face turn by `2 pi` times its index and every generator, a
    closed loop of vertices around a handle, by nothing; they come with the circulation and its target."""
    corner_normals = (mesh.incidence[2, 0] * normals).batch()   # [corners, faces] Vector-
    loop_normals = [(mesh.reading(loop[:, None]) * normals).batch() for loop in generators]   # [generators] list of [steps, 1] Vector-
    curvature = loop_turn(corner_normals)                                         # [faces] Scalar+
    # Around each generator, the extra turns undo the surface's own holonomy.
    undone = [-loop_turn(along) for along in loop_normals]                        # [generators] list of [1] Scalar+
    # The circulation of edge values around each face, then around each generator.
    circulation = mesh.circulation(generators)                                    # (Scalar+ <- Scalar+)[faces + generators <- edges]
    target = concatenate([2 * np.pi * as_scalar(indices) - curvature, *undone]).field()   # Scalar+[faces + generators]
    return circulation.lstsq(target), circulation, target

In [ ]:
# For each scene, the icosahedron's corners whose faces carry a singularity, and the index of each; corners 9
# and 10 lie opposite each other.
sphere_corners = [[9, 10], [9], [9, 4, 8, 3]]
sphere_indices = [[1, 1], [2], [1, 1, 1, -1]]

In [ ]:
sphere_singular = [sphere_mesh.vertex_faces[corners] for corners in sphere_corners]
sphere_designs = [trivial_connection(sphere_mesh, sphere_normals, face_indices(sphere_mesh, singular, indices), [])
                  for singular, indices in zip(sphere_singular, sphere_indices)]
residuals = [circulation(angles) - target for angles, circulation, target in sphere_designs]   # [scenes] list of Scalar+[faces]

## 7. The parallel field

The field itself is the null space of the connection Laplacian of the adjusted transport: `connection` of the adjusted rotors, with each edge weighted by its cotangent ratio. With a trivial connection, a field carried out from one vertex agrees with itself around every loop, so the Laplacian has two fields of energy zero, one a quarter turn of the other in every tangent plane, both of unit length everywhere. The stiff weight along the normals in `low_modes` keeps the normal bundle out of the bottom of the spectrum.

In [ ]:
def parallel_field(mesh: Mesh, normals: Vector, areas: Scalar, weights: Scalar, angles: Scalar) -> tuple[Scalar, Point]:
    """The two smoothest fields of the adjusted transport have their energies, and the first is taken as a unit
    field."""
    _, laplacian = connection(mesh, adjusted_transport(mesh, normals, angles), weights)   # (Point- <- Point-)[vertices <- vertices]
    energies, fields = low_modes(laplacian, normals, areas, normal_stiffness, 2)
    field = fields.batch()[0]                                                    # [vertices] Point-
    return energies, (field / field.norm()).field()                             # [2] Scalar+, Point-[vertices]


sphere_energies, sphere_fields = zip(*[parallel_field(sphere_mesh, sphere_normals, sphere_areas, sphere_weights, angles)
                                       for angles, _, _ in sphere_designs])
sphere_scenes = list(zip(sphere_fields, sphere_singular, sphere_indices))

save_figure(render.draw_fields(sphere_mesh, sphere_vertices, sphere_scenes), "connections_fields_spheres");

The surface grain follows the field continuously; its whorls and saddles reveal the singularities. On the left are two faces of index one at opposite sides of the sphere, one of them in view; in the middle, a single face of index two gathers the sphere's turning into one place; on the right, three faces of index one surround a saddle of index minus one. The dense end of each streak fades downstream, distinguishing sources from sinks.

## 8. Indices must add up to the Euler characteristic

Around every face the circulation of the extra turns adds up to the circulation of each edge counted once forwards and once backwards, nothing, so the face targets must add up to nothing too: the indices times `2 pi` must equal the curvature of the whole surface, which is the Euler characteristic times `2 pi`. That is the theorem of Poincaré and Hopf, and here it is the condition for the linear system to have a solution. Indices that fall short by one leave one whole turn unmet, which the least-squares solution spreads evenly over all the faces.

In the notation of discrete exterior calculus, the edge angles are a 1-form $\theta$, the face constraints read $d\theta = 2\pi k - K$ with $K$ the discrete Gaussian curvature, and the condition for a solution is $\sum k = \chi$.

In [ ]:
angles, circulation, target = trivial_connection(sphere_mesh, sphere_normals, face_indices(sphere_mesh, np.array([0]), [1]), [])
unmet = circulation(angles) - target                                            # Scalar+[faces]

print("the unmet turn, summed over the faces, in turns:", (unmet.sites.sum() / (2 * np.pi)).to_array())

## 9. The torus

On a torus the indices add up to zero, so a field with no singularities at all exists. Asking every face for zero is not enough to make the connection trivial, though: a loop that runs once around the tube, or once around the hole, encloses no face, and transport around it could still turn vectors. Those two loops are the torus's generators, and two more rows of the linear system ask each of them to turn by nothing, the extra turns undoing the surface's own holonomy around it. On the left, the torus with no singularities; on the right, a pair of singularities of opposite index on the side facing the viewer.

In [ ]:
# The torus of the designs, by its vertices around the hole and around the tube; for each scene the faces,
# by their place around the hole and around the tube, carrying a singularity, and the index of each.
n_ring_major, n_ring_minor = 40, 20
torus_places = [np.zeros((0, 2), dtype=int), np.array([[32, 2], [0, 2]])]
torus_indices = [[], [1, -1]]

In [ ]:
ring_mesh, ring_vertices = Mesh.torus(major_radius, minor_radius, n_ring_major, n_ring_minor)
ring_normals = vertex_normals(ring_mesh, ring_vertices)                              # Vector-[vertices]
ring_areas = vertex_areas(ring_mesh, ring_vertices)                                  # Scalar+[vertices]
ring_weights = cotangent_weights(ring_mesh, ring_vertices, as_scalar(np.ones(len(ring_mesh.faces))).field())   # Scalar+[edges]
# Its two generators: once around the tube, and once around the hole along the outer equator.
generators = [np.arange(n_ring_minor), np.arange(n_ring_major) * n_ring_minor]
torus_singular = [places[:, 0] * n_ring_minor + places[:, 1] for places in torus_places]
torus_designs = [trivial_connection(ring_mesh, ring_normals, face_indices(ring_mesh, singular, indices), generators)
                 for singular, indices in zip(torus_singular, torus_indices)]
torus_residuals = [circulation(angles) - target for angles, circulation, target in torus_designs]   # [scenes] list of Scalar+[faces + generators]
torus_energies, torus_fields = zip(*[parallel_field(ring_mesh, ring_normals, ring_areas, ring_weights, angles)
                                     for angles, _, _ in torus_designs])
torus_scenes = list(zip(torus_fields, torus_singular, torus_indices))

save_figure(render.draw_fields(ring_mesh, ring_vertices, torus_scenes), "connections_fields_tori");

## 10. Turning the field

The null space holds a field and its quarter turn, so any turn of the field by one angle about every normal is parallel too. As the field turns, the grain changes continuously between radial and circulating patterns around the visible singularity.

In [ ]:
# The frames of a whole turn.
n_turning_frames = 84

In [ ]:
turn_angles = np.linspace(0, 2 * np.pi, n_turning_frames, endpoint=False)        # [frames]
planes = sphere_normals.batch().dual()                                           # [vertices] Bivector+
field, singular, indices = sphere_scenes[0]
turned = ((planes * (-turn_angles[:, None] / 2)).exp() >> field.batch()).field()   # [frames] Point-[vertices]

show_animation(render.animate_turning(sphere_mesh, sphere_vertices, turned), "connections_turning", duration_ms)

## 11. Stripes

A stripe pattern is a field of evenly spaced stripes running across a surface at right angles to a chosen direction field: the stripes' phase advances along the field at a chosen frequency, as in the stripe patterns of Knöppel, Crane, Pinkall and Schröder. Across each edge the phase should advance by the frequency times the field's component along the edge, the edge's target, and the phase is carried by a tangent vector at every vertex, measured as its angle from the direction field. The direction field of a trivial connection is parallel for that connection, so a tangent vector carried by the same transport and turned further by the edge's target in the head's tangent plane has its angle from the field advance by exactly the target. The stripe field is the smoothest tangent field of that connection, the trivial connection's turns with the targets added: one more set of rotors for `connection`, with each face adding its half-cotangents to its edges' weights only where the direction field does not turn inside it, since across a singular face the field's average means nothing.

In [ ]:
# The finer sphere of the stripes, and the stripes' frequency on it.
n_stripe_subdivisions = 4
frequency = 20.0

In [ ]:
def stripes_along(mesh: Mesh, vertices: Point, normals: Vector, areas: Scalar, weights: Scalar, index: np.ndarray
                  ) -> tuple[tuple[Point, Point, Scalar], Scalar]:
    """One family of stripes runs along the parallel field of the trivial connection with every face turning by
    its index: the field's line, its unit stripe field, and the edges' targets, `[1] Point-[vertices]`,
    `[1] Point-[vertices]` and `[1] Scalar+[edges]`, with the energies of the two smoothest stripe fields."""
    angles, _, _ = trivial_connection(mesh, normals, index, [])
    _, field = parallel_field(mesh, normals, areas, weights, angles)             # Point-[vertices]
    # The phase advance across every edge: the frequency times the field averaged over the edge's ends, read along
    # the edge by the metric, the paper's inner product of the field with the edge.
    edge_means = mesh.incidence[1, 0].mean(axis=0)                               # Scalar+[edges <- vertices]
    advance = (edge_means * field).dot(mesh.d[0] * vertices) * frequency         # Scalar+[edges]
    # Every face is regular except the singular ones.
    regular = as_scalar(np.where(index == 0, 1.0, 0.0)).field()                 # Scalar+[faces]
    _, laplacian = connection(mesh, adjusted_transport(mesh, normals, angles + advance), cotangent_weights(mesh, vertices, regular))
    energies, stripe_fields = low_modes(laplacian, normals, areas, normal_stiffness, 2)
    stripe = stripe_fields[0]                                                    # Point-[vertices]
    return (field[None], (stripe / stripe.norm())[None], advance[None]), energies


fine_mesh, fine_vertices = Mesh.icosphere(n_stripe_subdivisions)
fine_normals = vertex_normals(fine_mesh, fine_vertices)                               # Vector-[vertices]
fine_areas = vertex_areas(fine_mesh, fine_vertices)                                   # Scalar+[vertices]
fine_weights = cotangent_weights(fine_mesh, fine_vertices, as_scalar(np.ones(len(fine_mesh.faces))).field())   # Scalar+[edges]
fine_singular = [fine_mesh.vertex_faces[corners] for corners in sphere_corners]
stripe_patterns, stripe_energies = zip(*[stripes_along(fine_mesh, fine_vertices, fine_normals, fine_areas, fine_weights,
                                                       face_indices(fine_mesh, singular, indices))
                                         for singular, indices in zip(fine_singular, sphere_indices)])

save_figure(render.draw_bands(fine_mesh, fine_vertices, stripe_patterns), "connections_stripes_spheres");

The light and dark bands are the stripes, from the stripe field's angle to the direction field, for the three designed fields of the sphere above. The stripe field comes as a pair, a field and its quarter turn, two patterns shifted by a quarter of a stripe. Where the targets around a face add up to more than the face's singularity allows, no phase can follow them all, the lowest eigenfield gives up there, and the stripes fork.

A cross field is a pair of lines at right angles at every vertex, which may turn by a quarter or a half turn around a singularity. A trivial connection whose faces turn by quarter turns gives one: a unit tangent vector carried out from the first vertex along a breadth-first tree, in one sparse solve, lands on every head turned by a whole number of quarter turns from the head's own. Stripes run across both lines, but carried across an edge each of the tail's lines lands on one of the head's lines, pointing along it or against it, and a stripe phase along a line reads backwards along the line reversed, so the stripe field carried onto a line it lands against is mirrored. The two families are one field over the cross's two lines at every vertex, coupled across the edges where the lines trade places, and its covariant derivative has three cells in each row. On the left, singularities at the eight corners of a cube, each turning by a quarter; on the right, at the four corners of a tetrahedron inscribed in it, each turning by a half.

In [ ]:
# The cross fields' singularities: at the cube's eight corners, each turning by a quarter, and at the four
# corners of a tetrahedron inscribed in it, each turning by a half.
cube = [[1, 1, 1], [1, 1, -1], [1, -1, 1], [1, -1, -1], [-1, 1, 1], [-1, 1, -1], [-1, -1, 1], [-1, -1, -1]]
cross_corners = [cube, [[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]]
cross_indices = [0.25, 0.5]
# The figure of the cross fields at a share of the other patterns' panel side.
cross_scale = 0.5

In [ ]:
def carried_out(mesh: Mesh, vertices: Point, normals: Vector, angles: Scalar) -> Point:
    """A unit tangent vector at every vertex, `Point-[vertices]`, is carried out from the first vertex by the
    transport of the given turns along a breadth-first tree: one sparse solve, each tree edge asking its head to
    be its tail carried across it, and the first vertex asking to point across its first edge. Where the turns
    make a trivial connection whose faces turn by quarter turns, this is a cross field: carried across any edge,
    the tail's vector lands on the head's turned by a whole number of quarter turns."""
    children, tree = mesh.spanning_tree
    turns = adjusted_transport(mesh, normals, angles)                            # [2] Rotor+[edges]
    covariant = (spdiag(turns >> Point) * mesh.boundary[1, 0]).sum(axis=0)       # (Point- <- Point-)[edges <- vertices]
    # Each tree edge's covariant derivative, in the row of the vertex it reaches.
    tree_edges = SparseExtensor.selection(context, tree, len(mesh.edges))       # Scalar+[vertices - 1 <- edges]
    along_tree = ~mesh.reading(children) * (tree_edges * covariant)             # (Point- <- Point-)[vertices <- vertices]
    first = mesh.reading(np.array([0]))                                          # Scalar+[1 <- vertices]
    first_edge = SparseExtensor.selection(context, np.array([0]), len(mesh.edges))   # Scalar+[1 <- edges]
    # The direction across the first edge, in the tangent plane: the plane through the normal and the edge's
    # direction, which is that direction as an antivector.
    seed = ((first * normals) ^ (first_edge * (mesh.d[0] * vertices)).dual_inverse()).normalized()   # Point-[1]
    return (along_tree + ~first * first * Point).solve(~first * seed)


def cross_stripes(mesh: Mesh, vertices: Point, normals: Vector, areas: Scalar, angles: Scalar, cross: Point,
                  regular: Scalar, frequency: float) -> tuple[Point, Point, Scalar]:
    """Two families of stripes run across the two lines of a cross field, `Point-[vertices]`, parallel up to
    quarter turns for the connection of the given turns: the lines `[2] Point-[vertices]`, the stripe field of
    each `[2] Point-[vertices]`, and the targets across every edge for each line at its head `[2] Scalar+[edges]`.

    Carried across an edge, each of the tail's lines lands on one of the head's lines, pointing along it
    or against it. A stripe phase along a line reads backwards along the line reversed, so the stripe
    field carried onto a line it lands against is mirrored: its component along that line reversed, the
    stripes' angle from the line negated. The two families are one field over the cross's two lines at
    every vertex, coupled across the edges where the lines trade places, around the cross's singularities.
    Each face adds its half-cotangents to its edges' weights only where it is regular, `Scalar+[faces]`."""
    count, n_edges = mesh.n_vertices, len(mesh.edges)
    forward = adjusted_transport(mesh, normals, angles)[0]                       # Rotor+[edges]
    # The cross's second line: the first turned a quarter turn by its commutator with the tangent plane.
    lines = stack([cross, cross.commutator(normals.dual())])                     # [2] Point-[vertices]
    tails, heads = mesh.incidence[1, 0]                                          # Scalar+[edges <- vertices], Scalar+[edges <- vertices]
    at_tails, ahead = tails * lines, heads * lines                               # [2] Point-[edges], [2] Point-[edges]
    # One where the tail's line b lands along the head's line a, minus one where it lands against it, and
    # zero where it lands across it.
    match = ahead[:, None].dot((forward >> at_tails)[None, :])                   # [2, 2] Scalar+[edges]
    behind = (match * at_tails[None, :]).sum(axis=1)                             # [2] Point-[edges]
    # The field read along each edge, by the metric: the paper's inner product of the field with the edge.
    advance = ((ahead + behind) / 2).dot(mesh.d[0] * vertices) * frequency       # [2] Scalar+[edges]
    target_turn = ((heads * normals).dual() * (-advance / 2)).exp()              # [2] Rotor+[edges]
    # Carried, then mirrored onto the head's line where it lands against it.
    moved = forward >> Point                                                     # (Point- <- Point-)[edges]
    along = ahead[:, None]                                                       # [2, 1] Point-[edges]
    landed = match * match * moved + (match - match * match) * (along * along.dot(moved))   # [2, 2] (Point- <- Point-)[edges]
    # The doubled field holds the two lines of every vertex as sites of their own, line a of vertex v at 2 v + a,
    # and the doubled edges likewise; these read line a at every vertex and at every edge.
    vertex_lines = SparseExtensor.selection(context, np.arange(2 * count).reshape(count, 2).T, 2 * count)   # [2] Scalar+[vertices <- vertices * 2]
    edge_lines = SparseExtensor.selection(context, np.arange(2 * n_edges).reshape(n_edges, 2).T, 2 * n_edges)   # [2] Scalar+[edges <- edges * 2]
    # The head's stripe field turned back by the target, against the tail's carried and mirrored.
    on_lines = (spdiag(target_turn.reverse() >> Point) * heads * vertex_lines
                - (spdiag(landed) * tails * vertex_lines[None]).sum(axis=1))     # [2] (Point- <- Point-)[edges <- vertices * 2]
    covariant = (~edge_lines * on_lines).sum(axis=0)                             # (Point- <- Point-)[edges * 2 <- vertices * 2]
    # Each line's site takes its edge's weight, and its vertex's normal and area.
    weights = (~edge_lines * cotangent_weights(mesh, vertices, regular)).sum(axis=0)   # Scalar+[edges * 2]
    laplacian = covariant.adjoint()(spdiag(weights) * covariant)                 # (Point- <- Point-)[vertices * 2 <- vertices * 2]
    _, stripe_fields = low_modes(laplacian, (~vertex_lines * normals).sum(axis=0),
                                 (~vertex_lines * areas).sum(axis=0), normal_stiffness, 1)
    stripe = vertex_lines * stripe_fields[0]                                     # [2] Point-[vertices]
    return lines, stripe / stripe.norm(), advance


cross_singular = [face_indices(fine_mesh, nearest_faces(fine_mesh, fine_vertices, mv.vector(corners)), quarters)
                  for corners, quarters in zip(cross_corners, cross_indices)]
cross_angles = [trivial_connection(fine_mesh, fine_normals, index, [])[0] for index in cross_singular]
cross_patterns = [cross_stripes(fine_mesh, fine_vertices, fine_normals, fine_areas, angles,
                                carried_out(fine_mesh, fine_vertices, fine_normals, angles),
                                as_scalar(np.where(index == 0, 1.0, 0.0)).field(), frequency)
                  for index, angles in zip(cross_singular, cross_angles)]

save_figure(render.draw_lines(fine_mesh, fine_vertices, cross_patterns, cross_scale), "connections_stripes_crosses");

In [ ]:
# checks
# Every vector along the latitude circles stays tangent and turns by the cap's solid angle.
carried = stack(tuple(transported(rings, starts)))                               # [steps + 1, rings] Point-
# A direction's pairing with a normal, the tangent plane's patch, is its part along the normal.
np.testing.assert_allclose((carried & rings).to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose(carried[-1].dot(starts).to_array(), np.cos(2 * np.pi * (1 - np.cos(colatitudes))), atol=2e-3)
# Neither heat diffusion nor the connection Laplacian's tangent modes leak into the normal direction.
np.testing.assert_allclose((diffused & torus_normals).to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose((modes & torus_normals).to_array(), 0.0, atol=1e-11)
# Quarter-turn rotation in each vertex's tangent plane commutes with the connection.
values = mode_energies.to_array()
np.testing.assert_allclose(values[0], values[1], rtol=1e-10)
np.testing.assert_allclose(values[2], values[3], rtol=1e-10)
weighting = spdiag(torus_weights)                                                # Scalar+[edges <- edges]
quarter_turned = modes.batch().commutator(normals.dual()).field()                # [modes] Point-[vertices]
mode_energy = (weighting * covariant(modes).norm_squared()).sites.sum().to_array()
turned_energy = (weighting * covariant(quarter_turned).norm_squared()).sites.sum().to_array()
np.testing.assert_allclose(mode_energy, turned_energy, rtol=1e-10)
# The faces' holonomies add up to two whole turns on the sphere.
np.testing.assert_allclose(curvature.sum(axis=0).to_array(), 4 * np.pi, rtol=1e-12)
# Every design meets its circulation, and its parallel fields have no energy.
np.testing.assert_allclose(np.concatenate([residual.to_array() for residual in residuals + torus_residuals]), 0.0, atol=1e-12)
np.testing.assert_allclose(np.concatenate([energies.to_array() for energies in [*sphere_energies, *torus_energies]]), 0.0, atol=1e-12)
# Indices one short of the Euler characteristic leave one whole turn spread over the faces.
np.testing.assert_allclose(unmet.to_array(), 2 * np.pi / len(sphere_mesh.faces), rtol=1e-9)
# Every stripe field comes with its quarter turn, of equal energy, and across most edges its phase, the
# angle from the stripe field to the direction field, advances by the edge's target.
for (lines, stripes, advance), energies in zip(stripe_patterns, stripe_energies):
    np.testing.assert_allclose(energies.to_array()[1], energies.to_array()[0], rtol=1e-6)
    # The sine of the angle from the stripe field to the direction field: the plane their commutator turns in,
    # read across the tangent plane; the cosine is their inner product.
    sines = (lines.commutator(stripes) | fine_normals.dual()).batch().to_array()   # [1, vertices]
    phase = as_scalar(np.arctan2(sines, stripes.dot(lines).batch().to_array())).field()   # [1] Scalar+[vertices]
    steps = (fine_mesh.d[0] * phase - advance).batch().to_array()                 # [1, edges]
    assert np.median(np.abs(np.angle(np.exp(1j * steps)))) < 0.25
# Carried across every edge, each of the tail's lines lands along or against one of the head's lines, and
# some edges trade the two lines or reverse them.
for (lines, _, _), angles in zip(cross_patterns, cross_angles):
    forward = adjusted_transport(fine_mesh, fine_normals, angles)[0]                  # Rotor+[edges]
    tails, heads = fine_mesh.incidence[1, 0][:, None] * lines                         # [2] Point-[edges], [2] Point-[edges]
    match = heads[:, None].dot((forward >> tails)[None, :]).batch().to_array()        # [2, 2, edges]
    np.testing.assert_allclose(np.abs(match) * (1 - np.abs(match)), 0, atol=1e-8)
    assert (match[0, 0] < 0.5).any()